# 12 — Capstone: Multimodal Document Q&A Pipeline

This capstone integrates: CLIP retrieval → layout-aware encoding → VLM cross-attention → generation with CFG → hallucination evaluation. The scenario is a document Q&A system that answers questions about a set of invoices and charts.

In [ ]:
# Multimodal Document Q&A Capstone
import numpy as np

def softmax(x, axis=-1):
    e = np.exp(x - x.max(axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)

def normalize(x):
    return x / (np.linalg.norm(x, axis=-1, keepdims=True) + 1e-8)

print('='*60)
print('Multimodal Document Q&A Pipeline')
print('='*60)

np.random.seed(42)
D = 128

# === 1. DOCUMENT KNOWLEDGE BASE ===
print('\n[1] DOCUMENT KNOWLEDGE BASE')
documents = [
    {'id': 'inv001', 'type': 'invoice',  'desc': 'Invoice #001, Total $1450, Widget order'},
    {'id': 'inv002', 'type': 'invoice',  'desc': 'Invoice #002, Total $850, Gadget order'},
    {'id': 'chart1', 'type': 'chart',    'desc': 'Q4 revenue bar chart, $2.3M'},
    {'id': 'chart2', 'type': 'chart',    'desc': 'Monthly trend line, 15% growth'},
    {'id': 'form1',  'type': 'form',     'desc': 'Employee onboarding form, John Smith'},
]
# Simulate CLIP document embeddings (doc type determines cluster)
inv_center   = np.random.randn(D)
chart_center = np.random.randn(D)
form_center  = np.random.randn(D)
centers = {'invoice': inv_center, 'chart': chart_center, 'form': form_center}
doc_embeds = {d['id']: normalize(centers[d['type']] + np.random.randn(D)*0.2) for d in documents}
print(f'  Indexed {len(documents)} documents in vector store')

In [ ]:
# === 2. QUERY RETRIEVAL ===
print('\n[2] QUERY RETRIEVAL (CLIP)')
queries = [
    ('What is the total on the widget invoice?', inv_center),
    ('Show me revenue trends', chart_center),
]

def retrieve(query_embed, doc_embeds_dict, k=2):
    q = normalize(query_embed)
    scores = {doc_id: float(np.dot(q, emb)) for doc_id, emb in doc_embeds_dict.items()}
    return sorted(scores.items(), key=lambda x: -x[1])[:k]

retrieved_docs = {}
for qtext, qemb in queries:
    qemb_noisy = qemb + np.random.randn(D)*0.15
    results = retrieve(qemb_noisy, doc_embeds, k=2)
    retrieved_docs[qtext] = results
    print(f'  Q: "{qtext}"')
    for doc_id, score in results:
        doc = next(d for d in documents if d['id'] == doc_id)
        print(f'    [{score:.4f}] {doc["id"]} - {doc["desc"]}')

# === 3. LAYOUT-AWARE ENCODING ===
print('\n[3] LAYOUT-AWARE ENCODING')
# Simulate LayoutLM encoding for retrieved invoice
inv_tokens = [
    ('Invoice', {'x0':50,'y0':20,'x1':200,'y1':50}),
    ('#001',    {'x0':210,'y0':20,'x1':300,'y1':50}),
    ('Total',   {'x0':300,'y0':200,'x1':400,'y1':225}),
    ('$1450',   {'x0':400,'y0':200,'x1':550,'y1':225}),
]
layout_feats = []
for text, bbox in inv_tokens:
    # Encode position: normalise bbox coords and embed
    pos_vec = np.array([bbox['x0'], bbox['y0'], bbox['x1'], bbox['y1']]) / 1000.0
    text_emb = np.random.randn(D//4)  # simulated text token embedding
    combined = np.concatenate([text_emb, np.tile(pos_vec, D//16)])
    layout_feats.append(combined[:D])
layout_matrix = np.array(layout_feats)
print(f'  Layout-encoded tokens: {layout_matrix.shape} (text + 2D position)')

In [ ]:
# === 4. VLM GENERATION (simulated) ===
print('\n[4] VLM RESPONSE GENERATION')

qa_pairs = [
    ('What is the total on the widget invoice?',  'The total on Invoice #001 for the Widget order is $1,450.00.'),
    ('Show me revenue trends',                    'The Q4 revenue bar chart shows $2.3M total, with the monthly trend line indicating 15% growth.'),
]
for q, a in qa_pairs:
    print(f'  Q: {q}')
    print(f'  A: {a}')
    print()

# === 5. HALLUCINATION CHECK ===
print('[5] HALLUCINATION CHECK (POPE-style)')
# Check if claimed entities actually exist in retrieved documents
claimed_entities = [
    ('$1,450.00', True,  'present in inv001'),
    ('Widget',    True,  'present in inv001'),
    ('$2.3M',     True,  'present in chart1'),
    ('15%',       True,  'present in chart2'),
    ('$999',      False, 'NOT in any document'),  # hallucination scenario
]
n_hall = 0
for entity, present, note in claimed_entities:
    status = 'GROUNDED' if present else 'HALLUCINATED'
    n_hall += not present
    print(f'  {entity:<15} {status}  ({note})')
print(f'\n  Hallucination rate: {n_hall}/{len(claimed_entities)} = {n_hall/len(claimed_entities):.1%}')

# === SUMMARY ===
print('\n' + '='*60)
print('PIPELINE SUMMARY')
print('='*60)
print(f'  Documents indexed:    {len(documents)}')
print(f'  Queries processed:    {len(queries)}')
print(f'  Retrieval method:     CLIP cosine similarity')
print(f'  Encoding:             LayoutLM-style (text + 2D bbox)')
print(f'  Generation:           VLM cross-attention (simulated)')
print(f'  Hallucination check:  POPE-style entity grounding')
print(f'  Grounding rate:       {(len(claimed_entities)-n_hall)/len(claimed_entities):.1%}')